# Telecom Italia (Trentino) – Step 1: Time Conversion, Aggregation and Cell Completeness

**Goal of this notebook (two simple transformations + one check)**

| Step | What happens | Why |
|---|---|---|
| 1. Time conversion | The raw time column is a Unix timestamp in **milliseconds (UTC)**. We convert it to a normal date/time in **local Italian time** (Europe/Rome). | The files have no date column, only this number. |
| 2. Aggregation | (a) Add up all **country codes** for the same cell and 10-minute slot. (b) Add up the six 10-minute slots of each hour → **hourly** data. | You asked for one record per cell and a one-hour time step. |
| 3. Completeness check | Count the cells and find the cells that do **not** have all 24 hours on every day. | To select clean cells for modelling. |

**Raw file format** (tab-separated, no header, 8 columns):
`CellID | time (ms since 1970-01-01 UTC) | country code | SMS-in | SMS-out | Call-in | Call-out | Internet`

Empty values mean *no activity was recorded* for that item.

Each day is saved to Google Drive after it is processed, so if Colab disconnects you can simply re-run and it continues where it stopped.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # Colab will ask you to allow access

## 2. Settings

Edit only the block between the `=====` lines.
`COLS` names the 8 raw columns. `ACT` lists the activity columns that are added together. The country code is not in `USE_COLS`, so it is simply left out when we add up the countries.

In [ ]:
import os, re, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

# ============ EDIT THESE ============
DATA_DIR  = '/content/drive/MyDrive/telecom_italia/trentino_raw'   # folder with the 62 .txt files
OUT_DIR   = '/content/drive/MyDrive/telecom_italia/processed'      # results are saved here
FILE_GLOB = '*.txt'
LOCAL_TZ  = 'Europe/Rome'      # time zone used to convert the timestamps
MIN_COMPLETENESS = 95.0        # % of hours present, used for the "relaxed" clean-cell list
# ====================================

COLS     = ['cell_id', 'ts_ms', 'country_code', 'sms_in', 'sms_out', 'call_in', 'call_out', 'internet']
ACT      = ['sms_in', 'sms_out', 'call_in', 'call_out', 'internet']   # columns that are summed
USE_COLS = ['cell_id', 'ts_ms'] + ACT                                  # country_code is dropped (= merged)

os.makedirs(f'{OUT_DIR}/daily_hourly', exist_ok=True)

## 3. List the raw files

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_DIR, FILE_GLOB)))

def file_date(p):
    # the day is only in the FILE NAME (e.g. ...-2013-11-01.txt), not inside the file
    m = re.search(r'(\d{4}-\d{2}-\d{2})', os.path.basename(p))
    return pd.Timestamp(m.group(1)) if m else pd.NaT

inv = pd.DataFrame({'path': files})
inv['file']    = inv['path'].map(os.path.basename)
inv['date']    = inv['path'].map(file_date)
inv['size_mb'] = inv['path'].map(lambda p: round(os.path.getsize(p) / 1e6, 1))

print(f'Files found : {len(inv)}  (expected 62)')
print(f'Total size  : {inv.size_mb.sum():,.0f} MB')
print(f'Date range  : {inv.date.min().date()} -> {inv.date.max().date()}')
print('Missing days:', [d.date().isoformat() for d in pd.date_range(inv.date.min(), inv.date.max()).difference(inv['date'])] or 'none')
inv.head()

### Valid time window

We know from the file names which days we expect (about 2013-11-01 to 2014-01-01). Any record whose timestamp falls far outside that window is a **bad timestamp** (for example a `0`, which would turn into 1970-01-01). The window below is used in the next steps to catch such records instead of letting them silently create a 1970 date.

In [ ]:
def to_ms(ts):                       # pandas Timestamp (UTC) -> milliseconds since 1970
    return int(ts.tz_localize('UTC').value // 10**6)

# one day of margin on both sides of the file dates (UTC vs local time at midnight)
TS_MIN_MS = to_ms(inv['date'].min() - pd.Timedelta(days=1))
TS_MAX_MS = to_ms(inv['date'].max() + pd.Timedelta(days=2))
print('Valid raw timestamps:', TS_MIN_MS, '->', TS_MAX_MS)

## 4. Look at one raw file (check the format)

In [ ]:
sample = files[0]
with open(sample) as f:
    first_lines = [f.readline() for _ in range(4)]
for l in first_lines:
    print(repr(l))
print('Fields in first line:', len(first_lines[0].rstrip('\n').split('\t')), '(expected 8)')

def has_header(path):                # True if the first value of the first line is not a number
    with open(path) as f:
        return not f.readline().split('\t')[0].strip().isdigit()

print('Header row detected:', has_header(sample))

In [ ]:
raw = pd.read_csv(sample, sep='\t', header=None, names=COLS, skiprows=1 if has_header(sample) else 0)
print('Rows                :', f'{len(raw):,}')
print('Unique cells        :', raw.cell_id.nunique())
print('Unique country codes:', raw.country_code.nunique())
print('Unique 10-min slots :', raw.ts_ms.nunique(), '(144 expected for a full day)')

# Time conversion on the sample: raw number -> UTC -> Italian local time
t = pd.to_datetime(raw.ts_ms.agg(['min', 'max']), unit='ms', utc=True).dt.tz_convert(LOCAL_TZ)
print('First slot (local):', t['min'], '| last slot (local):', t['max'])

print('Share of empty values per column (%):')
print((raw.isna().mean() * 100).round(1).to_dict())
raw.head()

## 5. (Optional) Scan all raw files for bad timestamps

This reads only a few columns of every file and lists any record whose time value is outside the valid window. It tells you **which file** contains such records, what the raw value is, and how much traffic they carry. Skip it if you prefer; the processing step below also removes such records automatically.

In [ ]:
bad_parts = []
for p in tqdm(files, desc='Scanning raw files'):
    t = pd.read_csv(p, sep='\t', header=None, names=COLS,
                    usecols=['cell_id', 'ts_ms', 'country_code', 'internet'],
                    skiprows=1 if has_header(p) else 0)
    b = t[(t.ts_ms < TS_MIN_MS) | (t.ts_ms > TS_MAX_MS) | t.ts_ms.isna()]
    if len(b):
        bad_parts.append(b.assign(file=os.path.basename(p)))

if bad_parts:
    bad_rows = pd.concat(bad_parts)
    bad_rows.to_csv(f'{OUT_DIR}/bad_timestamp_rows.csv', index=False)
    print('Records with bad timestamps:', len(bad_rows))
    print('Distinct raw time values   :', bad_rows.ts_ms.unique()[:20])
    print('Cells affected             :', bad_rows.cell_id.nunique())
    print('Records per file:'); print(bad_rows.groupby('file').size())
    bad_rows.head(10)
else:
    print('No bad timestamps found in the raw files.')

## 6. Process every day: time conversion + aggregation

`process_file()` does the real work for one day:

1. **Read** the file (the country code column is skipped).
2. **Remove** records whose timestamp is outside the valid window (counted and reported).
3. **Merge country codes**: add up all rows with the same `(cell_id, ts_ms)`.
4. **Convert time**: raw milliseconds → UTC → Europe/Rome local time → round **down** to the hour (e.g. 14:30 and 14:50 both become 14:00).
5. **Hourly aggregation**: add up the 10-minute values of each `(cell, hour)`.
6. **`n_slots`**: how many different 10-minute slots had a record in that hour (6 = completely observed).

> In this dataset an empty value means no activity was logged, so empty values count as 0 when summed. A whole hour with no record at all stays "missing" (no row) and is handled in the completeness check.

In [ ]:
def process_file(path):
    # 1. read (skip the country code column)
    df = pd.read_csv(path, sep='\t', header=None, names=COLS, usecols=USE_COLS,
                     skiprows=1 if has_header(path) else 0)

    # 2. drop records with impossible timestamps (they would become 1970 dates)
    ok = df['ts_ms'].between(TS_MIN_MS, TS_MAX_MS)
    n_bad = int((~ok).sum())
    df = df[ok].copy()
    df['cell_id'] = df['cell_id'].astype('int32')

    # 3. merge country codes -> one record per cell and 10-min slot
    g = df.groupby(['cell_id', 'ts_ms'], sort=False)[ACT].sum().reset_index()

    # 4. time conversion (done on the ~144 distinct timestamps only, which is fast)
    uniq  = g['ts_ms'].unique()
    local = pd.to_datetime(uniq, unit='ms', utc=True).tz_convert(LOCAL_TZ).tz_localize(None)
    g['hour'] = g['ts_ms'].map(dict(zip(uniq, local))).dt.floor('h')

    # 5-6. 10-minute -> hourly (sum of activity, plus count of 10-min slots present)
    agg = {c: 'sum' for c in ACT}
    agg['ts_ms'] = 'size'
    h = (g.groupby(['cell_id', 'hour'], sort=False).agg(agg)
           .rename(columns={'ts_ms': 'n_slots'}).reset_index())
    h[ACT] = h[ACT].astype('float32')          # smaller memory
    h['n_slots'] = h['n_slots'].astype('int8')
    return h, n_bad

bad_log = {}
for p in tqdm(files, desc='Processing days'):
    out = os.path.join(OUT_DIR, 'daily_hourly', os.path.basename(p).rsplit('.', 1)[0] + '.parquet')
    if os.path.exists(out):               # already processed -> skip (lets you resume)
        continue
    h, n_bad = process_file(p)
    h.to_parquet(out, index=False)        # save this day to Drive
    if n_bad:
        bad_log[os.path.basename(p)] = n_bad

print('Daily hourly files on Drive:', len(os.listdir(f'{OUT_DIR}/daily_hourly')))
print('Records removed for bad timestamps (this run):', bad_log or 'none')

## 7. Combine all days into one hourly table

All the small daily files are joined into one table `hourly` with one row per **cell and hour**.
The last two checks are safety nets: rows outside the file dates are removed (this also cleans daily files created by an earlier version of this notebook), and an hour split across two files would be merged.

In [ ]:
parts  = sorted(glob.glob(f'{OUT_DIR}/daily_hourly/*.parquet'))
hourly = pd.concat([pd.read_parquet(p) for p in tqdm(parts, desc='Loading')], ignore_index=True)

# Safety net 1: keep only hours inside the date range of the raw files
d0 = inv['date'].min()
d1 = inv['date'].max() + pd.Timedelta(days=1)
out_of_range = (hourly['hour'] < d0) | (hourly['hour'] >= d1)
print(f'Hourly rows outside {d0.date()} -> {(d1 - pd.Timedelta(days=1)).date()} removed: {out_of_range.sum():,}')
if out_of_range.any():
    print(hourly.loc[out_of_range, 'hour'].dt.date.value_counts().head(10))
hourly = hourly[~out_of_range]

# Safety net 2: an hour split across two files would appear twice -> merge
dups = hourly.duplicated(['cell_id', 'hour']).sum()
print('Duplicate (cell, hour) rows:', dups)
if dups:
    agg = {c: 'sum' for c in ACT}; agg['n_slots'] = 'sum'
    hourly = hourly.groupby(['cell_id', 'hour'], as_index=False).agg(agg)

hourly = hourly.sort_values(['cell_id', 'hour']).reset_index(drop=True)
print(f'Hourly table: {len(hourly):,} rows, {hourly.memory_usage(deep=True).sum() / 1e6:,.0f} MB')
hourly.head()

## 8. How many cells (square IDs)?

In this dataset *CellID* and *SquareID* are the same thing. We also build the list of all expected hours (`all_hours`) so that we know exactly how many hours every cell should have.

In [ ]:
hourly['date'] = hourly['hour'].dt.normalize()
all_days  = pd.date_range(hourly['date'].min(), hourly['date'].max(), freq='D')
all_hours = pd.date_range(hourly['hour'].min(), hourly['hour'].max(), freq='h')
EXPECTED_HOURS = len(all_hours)

print(f'Unique cells (square IDs)  : {hourly.cell_id.nunique():,}')
print(f'Cell ID range              : {hourly.cell_id.min()} -> {hourly.cell_id.max()}')
print(f'Days covered               : {len(all_days)}   ({all_days.min().date()} -> {all_days.max().date()})')
print(f'Expected hours per cell    : {EXPECTED_HOURS}  ({len(all_days)} days x 24)')

cells_per_day = hourly.groupby('date').cell_id.nunique()
print(f'Cells reporting per day    : min {cells_per_day.min():,} | max {cells_per_day.max():,}')

ax = cells_per_day.plot(figsize=(11, 3.5), marker='o', ms=3, title='Number of cells with data per day')
ax.set_ylabel('cells'); plt.show()

## 9. Which cells do not have all 24 hours?

`cd` is a table with one row per cell and one column per day, holding the **number of hours present** that day (24 = complete). From it we build `report` with one row per cell:

- `days_full_24h`: days on which the cell has all 24 hours
- `hours_present` / `hours_missing` / `completeness_pct`: over the whole period
- `hours_all_6_slots`: hours where all six 10-minute slots were recorded (stricter)
- `complete_all_days`: True if the cell has all 24 hours on **every** day

In [ ]:
cd = (hourly.groupby(['cell_id', 'date']).size().unstack('date')
            .reindex(columns=all_days).fillna(0).astype('int16'))

report = pd.DataFrame(index=cd.index)
report['days_present']     = (cd > 0).sum(axis=1)
report['days_full_24h']    = (cd == 24).sum(axis=1)
report['hours_present']    = cd.sum(axis=1)
report['hours_missing']    = EXPECTED_HOURS - report['hours_present']
report['completeness_pct'] = (100 * report['hours_present'] / EXPECTED_HOURS).round(2)

full6 = hourly[hourly.n_slots == 6].groupby('cell_id').size()
report['hours_all_6_slots'] = full6.reindex(report.index).fillna(0).astype(int)

report['complete_all_days'] = report['days_full_24h'] == len(all_days)
report.index.name = 'cell_id'

n_total    = len(report)
n_complete = int(report.complete_all_days.sum())
print(f'Total cells                                   : {n_total:,}')
print(f'Cells with all 24 h on ALL {len(all_days)} days            : {n_complete:,}  ({100*n_complete/n_total:.1f} %)')
print(f'Cells with at least one missing hour          : {n_total - n_complete:,}  ({100*(n_total-n_complete)/n_total:.1f} %)')
print(f'Cells with >= {MIN_COMPLETENESS:.0f}% of hours present        : {(report.completeness_pct >= MIN_COMPLETENESS).sum():,}')

In [ ]:
incomplete = report[~report.complete_all_days].sort_values('completeness_pct')
print(f'{len(incomplete):,} incomplete cells (worst first):')
display(incomplete.head(20))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
report.completeness_pct.hist(bins=50, ax=ax[0])
ax[0].set_title('Completeness per cell (% of expected hours)'); ax[0].set_xlabel('%'); ax[0].set_ylabel('cells')
report.hours_missing[report.hours_missing > 0].hist(bins=50, ax=ax[1])
ax[1].set_title('Missing hours per incomplete cell'); ax[1].set_xlabel('hours missing')
plt.tight_layout(); plt.show()

In [ ]:
# Which exact hours are missing for one cell?
def missing_hours(cell_id):
    present = pd.DatetimeIndex(hourly.loc[hourly.cell_id == cell_id, 'hour'])
    return all_hours.difference(present)

if len(incomplete):
    example = incomplete.index[len(incomplete) // 2]
    mh = missing_hours(example)
    print(f'Cell {example}: {len(mh)} missing hours, first ones:')
    print(mh[:10])

# At which hours of the day do the gaps occur most often (incomplete cells only)?
exp_per_hod = pd.Series(all_hours.hour).value_counts().sort_index()
got_per_hod = hourly[hourly.cell_id.isin(incomplete.index)].groupby(hourly.hour.dt.hour).size()
miss_rate = (1 - got_per_hod / (exp_per_hod * len(incomplete))).mul(100)
miss_rate.plot(kind='bar', figsize=(11, 3.2), title='Share of missing hours by hour of day (incomplete cells, %)')
plt.show()

## 10. Choose the clean cells and save

- **strict**: all 24 hours present on every day
- **relaxed**: at least `MIN_COMPLETENESS` % of hours present

Look at the counts and decide. A missing hour often means *no activity*, so quiet cells can look "incomplete" even though nothing is wrong with the data.

In [ ]:
clean_strict  = report.index[report.complete_all_days]
clean_relaxed = report.index[report.completeness_pct >= MIN_COMPLETENESS]
print(f'Strict  clean cells: {len(clean_strict):,}')
print(f'Relaxed clean cells: {len(clean_relaxed):,}')

report.to_csv(f'{OUT_DIR}/cell_completeness_report.csv')
pd.Series(clean_strict,  name='cell_id').to_csv(f'{OUT_DIR}/clean_cells_strict.csv',  index=False)
pd.Series(clean_relaxed, name='cell_id').to_csv(f'{OUT_DIR}/clean_cells_relaxed.csv', index=False)
pd.Series(incomplete.index, name='cell_id').to_csv(f'{OUT_DIR}/incomplete_cells.csv', index=False)

hourly.drop(columns='date').to_parquet(f'{OUT_DIR}/hourly_all_cells.parquet', index=False)
hourly[hourly.cell_id.isin(clean_strict)].drop(columns='date').to_parquet(f'{OUT_DIR}/hourly_clean_strict.parquet', index=False)
print('Saved to', OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))

In [ ]:
# Optional: full hourly grid for chosen cells. Missing hours are added as rows with 0 activity (n_slots = 0).
def build_full_grid(cell_ids):
    cell_ids = sorted(cell_ids)
    sub = hourly[hourly.cell_id.isin(cell_ids)].drop(columns='date').set_index(['cell_id', 'hour'])
    idx = pd.MultiIndex.from_product([cell_ids, all_hours], names=['cell_id', 'hour'])
    grid = sub.reindex(idx)
    grid[ACT] = grid[ACT].fillna(0).astype('float32')
    grid['n_slots'] = grid['n_slots'].fillna(0).astype('int8')
    return grid.reset_index()

# grid_relaxed = build_full_grid(clean_relaxed)
# grid_relaxed.to_parquet(f'{OUT_DIR}/hourly_clean_relaxed_filled.parquet', index=False)

## 11. Sanity check: daily pattern of Internet activity

In [ ]:
sel = hourly[hourly.cell_id.isin(clean_relaxed)]
prof = sel.groupby(sel.hour.dt.hour)['internet'].mean()
ax = prof.plot(figsize=(10, 3.2), marker='o', title='Mean hourly Internet activity by hour of day (relaxed clean cells)')
ax.set_xlabel('hour of day'); ax.set_xticks(range(0, 24)); plt.show()

sel.groupby('hour')['internet'].sum().plot(figsize=(12, 3.2), lw=0.8,
    title='Total hourly Internet activity (relaxed clean cells)')
plt.show()

## Next steps
- Choose strict vs relaxed clean cells from the counts in section 9
- EDA: daily/weekly seasonality, holidays (Dec 8, Dec 25, Jan 1), load distribution across cells
- Feature engineering: hour, weekday, holiday flag, lags, rolling statistics, SMS/call features
- Forecasting: seasonal naive baseline, SARIMA/ETS, then ML models
- Capacity scenarios (e.g. capacity = P95/P99 of history × headroom) and risk flags